# Target-indication pair tables

One row per target-indication pair, with the maximum clinical phase reached and the genetic
support propagated through the disease ontology. Oncology indications (descendants of
`MONDO_0045024`) are removed. Two support columns are carried: `score_all` from every
L2G-prioritised credible set, and `score_pav` from those containing a protein-altering
variant. Methods "Clinical trials success modelling".

Writes `ti_pairs_chembl`, `l2g_indirect_assoc_all`, `l2g_indirect_assoc_pav`, and, if the
Pharmaprojects table is available, `ti_pairs_pharmaprojects`.

In [ ]:
import numpy as np
import pandas as pd
from gentropy.common.session import Session
from gentropy.dataset.study_index import StudyIndex
from gentropy.dataset.study_locus import StudyLocus
from gentropy.method.drug_enrichment_from_evid import chemblDrugEnrichment
from pyspark.sql import functions as f

from manuscript_methods import paper
from manuscript_methods.enrichment import or_rs, support_mask

session = Session(extended_spark_conf={"spark.driver.memory": "40G"})

In [ ]:
sl = StudyLocus.from_parquet(session, paper.release("credible_set"))
si = StudyIndex.from_parquet(session, paper.release("study"))
disease_index = session.spark.read.parquet(paper.release("disease") + "/disease.parquet")
chembl_evidence = session.spark.read.parquet(paper.release("evidence") + "/sourceId=chembl")

l2g = session.spark.read.parquet(paper.derived("prioritised_genes_diseases"))
genes = session.spark.read.parquet(paper.derived("gene_table"))
print("L2G rows:", l2g.count(), "genes with pleiotropy:", genes.count())

## ChEMBL target-indication pairs, oncology removed

In [ ]:
efo_to_remove = chemblDrugEnrichment.selecting_all_decendands_based_on_efo_list(
    disease_index_orig=disease_index, efo_ids=["MONDO_0045024"]
)
chembl = chemblDrugEnrichment.process_chembl_evidence(chembl_evidence, efo_to_remove).cache()
n_chembl = chembl.count()
print("ChEMBL T-I pairs:", n_chembl)
chembl.groupBy("maxClinicalPhase").count().orderBy("maxClinicalPhase").show()

## Genetic support propagated through the ontology

In [ ]:
def indirect_assoc(table):
    """Propagate the L2G scores of one credible-set subset through the disease ontology."""
    evidence = chemblDrugEnrichment.to_disease_target_evidence(
        table_with_score=table.drop("diseaseIds"),
        score_column="score",
        datasource_id="l2g",
        study_locus=sl,
        study_index=si,
        min_score=0.1,
    )
    return chemblDrugEnrichment.evidence_to_indirect_assosiations(
        evidence, disease_index, use_max=True, efo_to_remove=efo_to_remove
    ).cache()


assoc_all = indirect_assoc(l2g)
assoc_pav = indirect_assoc(l2g.filter(f.col("VEP") == 1))
print("target-disease pairs with support (all):", assoc_all.count(), "(PAV):", assoc_pav.count())

for name, table in [("all", assoc_all), ("pav", assoc_pav)]:
    assert table.count() == table.select("targetId", "diseaseId").distinct().count(), name
assoc_all.write.mode("overwrite").parquet(paper.derived("l2g_indirect_assoc_all"))
assoc_pav.write.mode("overwrite").parquet(paper.derived("l2g_indirect_assoc_pav"))

## Master table

In [ ]:
gene_features = genes.select(
    f.col("geneId").alias("targetId"), "uniqueTherapeuticAreas", "uniqueDiseases", "approvedSymbol"
)

master = (
    chembl.join(assoc_all.withColumnRenamed("indirect_assoc_score", "score_all"), ["targetId", "diseaseId"], "left")
    .join(assoc_pav.withColumnRenamed("indirect_assoc_score", "score_pav"), ["targetId", "diseaseId"], "left")
    .join(gene_features, "targetId", "left")
    .toPandas()
)
assert len(master) == n_chembl, "joins changed the number of ChEMBL pairs"

master["in_gps"] = master["uniqueTherapeuticAreas"].notna()
master["approved"] = (master["maxClinicalPhase"] >= 4).astype(int)
master.to_parquet(paper.derived("ti_pairs_chembl"), index=False)

print("pairs:", len(master))
print("with any genetic support:", int(master["score_all"].notna().sum()))
print("with PAV genetic support:", int(master["score_pav"].notna().sum()))
print("approved:", int(master["approved"].sum()))

## Control: the two headline enrichment estimates

In [ ]:
checks = pd.DataFrame(
    [
        {"definition": "all GWAS", **or_rs(support_mask(master), master["approved"])},
        {"definition": "PAV + 2-5 TA", **or_rs(support_mask(master, pav=True, ta_min=2, ta_max=5), master["approved"])},
    ]
).set_index("definition")
print(checks[["odds_ratio", "relative_success", "yes_evid-high_clinphase", "p_value"]].to_string())
print()
print("manuscript: all GWAS OR 3.62 with 242 approved; PAV + 2-5 TA OR 10.3, RS 4.8, 51 approved")

## Pharmaprojects

Used by Supplementary Results 11 as an independently curated comparison. The processed table
comes from `chapters/_legacy/05-other-drug-indication-data`; it is a licensed resource and is
skipped when absent.

In [ ]:
from pathlib import Path

pharmaprojects = Path(paper.baseline("minikel_etal_processed_data_v2.csv"))
if not pharmaprojects.exists():
    print("Pharmaprojects table absent, skipping (GAPS.md)")
else:
    columns = [
        "targetId",
        "diseaseId",
        "meshId",
        "approvedSymbol",
        "ti_uid",
        "ccatnum",
        "maxClinicalPhase",
        "outcome",
        "geneticSupport_old",
        "genetic_insight",
        "target_status",
        "year_launch",
        "orphan",
    ]
    pp = pd.read_csv(pharmaprojects, low_memory=False)[columns].copy()
    pp_master = (
        pp.merge(
            assoc_all.toPandas().rename(columns={"indirect_assoc_score": "score_all"}),
            on=["targetId", "diseaseId"],
            how="left",
        )
        .merge(
            assoc_pav.toPandas().rename(columns={"indirect_assoc_score": "score_pav"}),
            on=["targetId", "diseaseId"],
            how="left",
        )
        .merge(gene_features.toPandas().drop(columns=["approvedSymbol"]), on="targetId", how="left")
    )
    assert len(pp_master) == len(pp), "joins changed the number of Pharmaprojects pairs"
    pp_master["in_gps"] = pp_master["uniqueTherapeuticAreas"].notna()
    pp_master["approved"] = pp_master["outcome"].astype(int)
    pp_master.to_parquet(paper.derived("ti_pairs_pharmaprojects"), index=False)
    print("Pharmaprojects pairs:", len(pp_master), "launched:", int(pp_master["approved"].sum()))
    # Their own genetic-support flag against their own outcome, as a provenance check.
    own = or_rs(pp_master["geneticSupport_old"].astype(bool), pp_master["approved"])
    print(
        "their flag: OR",
        round(own["odds_ratio"], 3),
        "log-odds ~ 0.843 in Minikel et al.:",
        round(np.log(own["odds_ratio"]), 4),
    )